In [18]:
import torch
from torch import nn

torch.__version__
device = "cuda" if torch.cuda.is_available() else "cpu"
device
import requests
import zipfile
from pathlib import Path
import os
import cv2
import glob
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
device
# import packages

import argparse
import time
import math
import random
import shutil
import sys
import glob

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import transforms


from pathlib import Path

from PIL import Image
from torch.utils.data import Dataset

In [19]:
seed = 123                                        # for reproducibility
cuda = True                                       # use GPU
save = True                                       # save trained model
image_dataset = r'C:\Users\Tassili\Desktop\GP\autoEncoder\driv\train'  # path to the root of the image dataset
sequence_dataset = r'C:\Users\Tassili\Desktop\GP\autoEncoder\driv\Video'  # path to the root of the video dataset
checkpoint = r'C:\Users\Tassili\Desktop\GP\autoEncoder\model1\test\checkpoint_best_loss.pth.tar'                                   # load pretrained model
epochs = 5                                       # total training epochs
clip_max_norm = 1.0                               # avoid gradient explosion
patch_size = (512, 512)                           # input size for the training network
learning_rate = 1e-3
batch_size = 16
test_batch_size = 16
num_workers = 2                         # multi-process for loading training data
N = 45
M = 60

In [20]:
# network structure define
def conv(in_channels, out_channels, kernel_size=6, stride=2):
    return nn.Conv2d(
        in_channels,
        out_channels,
        kernel_size=kernel_size,
        stride=stride,
        padding=kernel_size // 2,
    )


def deconv(in_channels, out_channels, kernel_size=6, stride=2):
    return nn.ConvTranspose2d(
        in_channels,
        out_channels,
        kernel_size=kernel_size,
        stride=stride,
        output_padding=stride - 1,
        padding=kernel_size // 2,
    )


class Network(nn.Module):

    def __init__(self,N, M, init_weights=True, **kwargs):
        super().__init__(**kwargs)

        self.g_a = nn.Sequential(
            conv(3, N),
            # nn.PReLU(),
            # nn.Conv2d(40, 40, kernel_size=1),
            nn.Conv2d(N, N, kernel_size=1),
            conv(N, N),
            nn.Conv2d(N, N, kernel_size=1),
            conv(N, N),
            nn.Conv2d(N, N, kernel_size=1),
            # nn.PReLU(),
            conv(N, 44),
        )


        self.g_s = nn.Sequential(
            deconv(44, N),
            deconv(N, N),
            deconv(N, N),
            nn.ConvTranspose2d(N, 3, kernel_size=3, stride=2, padding =2 , output_padding=1),

        )

        self.N = N
        self.M = M

        if init_weights:
            self._initialize_weights()

    def forward(self, x):
        y = self.g_a(x)
        x_hat = self.g_s(y)
        return {
            "x_hat": x_hat,
            # "x_quan": quan,
        }


    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def compress(self, x):
        y = self.g_a(x)
        return y

    def decompress(self, y_hat):
        x_hat = self.g_s(y_hat).clamp_(0, 1) # Limiting. Limit the value of input to [min, max], output as a tensor
        return {"x_hat": x_hat}



In [21]:
# change the type of images in the test set to tensors
import torch
from torchvision import transforms
transform1 = transforms.Compose([
	transforms.CenterCrop((512,512)),
	transforms.ToTensor(),
	]
)


In [22]:


device = "cuda" if cuda and torch.cuda.is_available() else "cpu"
net = Network(N,M)
net = net.to(device)


checkpoint = torch.load('checkpoint90percent_new_dataset.pth.tar', map_location=device)

# Get the state dictionary from the checkpoint
checkpoint_state_dict = checkpoint['state_dict']

# Load the model state dictionary partially with strict=False
net.load_state_dict(checkpoint_state_dict, strict=False)


<All keys matched successfully>

In [23]:
import time
from torchvision.transforms.functional import to_pil_image
from PIL import Image


img_PIL_Ten = img = transform1(Image.open((r'img1.png')))


ima = img_PIL_Ten.unsqueeze(0).to(device)



comp = net.compress(ima).squeeze(0).detach()


comp = torch.tensor(comp, dtype=torch.float16)



# Save the 'comp' variable to the file
torch.save(comp, "compressed_data.pt")




loaded_comp = torch.load("compressed_data.pt")
loaded_comp = torch.tensor(loaded_comp, dtype=torch.float32)





dec = net.decompress(loaded_comp)





imageRrcTen = dec['x_hat']

# Convert the tensor to a PIL Image using torchvision.transforms.functional
image_pil = to_pil_image(imageRrcTen)

# Save the PIL Image as a JPEG file
image_pil.save('output.jpg')






C:\Users\Tassili\AppData\Local\Temp\ipykernel_7572\4155751226.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  comp = torch.tensor(comp, dtype=torch.float16)
C:\Users\Tassili\AppData\Local\Temp\ipykernel_7572\4155751226.py:27: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loaded_comp = torch.tensor(loaded_comp, dtype=torch.float32)
